# EXP 10: LEX and YACC — Expression Evaluation and Infix→Postfix — Compiler Design lab
**Platform:** Google Colab • **Tools:** Flex, Bison, GCC

### Aim
- **10A:** To write and execute a LEX and YACC program to evaluate an arithmetic expression using the grammar `E -> E+E | E-E | E*E | E/E | E%E | (E) | number`.
- **10B:** To write and execute a LEX and YACC program to convert an infix expression to a postfix expression using the grammar `E -> E+T | E-T | T`, `T -> T*F | T/F | F`, `F -> (E) | -F | id`.

### Learning outcomes
After this lab, you can attach semantic values to grammar symbols with `%union`, carry a value from the scanner to the parser in `yylval`, evaluate an expression while parsing it, and build a postfix string bottom-up during reductions.


## 1. Semantic values: the new idea in this experiment
Exp 9 only *checked* the input. Now every grammar symbol also carries a **semantic value** that the actions compute with.

| Item | Role |
|---|---|
| `%union { int value; }` | Declares the type of the parser's value stack (each stack slot holds one value) |
| `%token <value> NUMBER` | NUMBER's semantic value is an `int`, stored in union member `value` |
| `%type <value> E` | Nonterminal E also stores an `int` in `value` |
| `yylval.value = atoi(yytext)` | Scanner stores the lexeme's numeric value **before** returning NUMBER |
| `$1`, `$3` | Semantic values of the 1st and 3rd symbols in the rule body |
| `$$` | The semantic value produced for the rule's left-hand side |
| `%type <str> E T F` | In Part B each symbol instead carries a `char[256]` postfix string |


## 2. Install the tools
Run this once per Colab runtime. On a Windows machine with Flex, Bison and GCC already on PATH this cell is unnecessary — its `apt-get` call raises a `FileNotFoundError`, which you can ignore or skip.

In [ ]:
import subprocess
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "flex", "bison", "gcc"], check=True)
for tool in ["flex", "bison", "gcc"]:
    print(subprocess.check_output([tool, "--version"], text=True).splitlines()[0])

## 3. Part A — evaluate an arithmetic expression
### Grammar
```text
E -> E+E | E-E | E*E | E/E | E%E | (E) | number
```
Precedence: `%left '+' '-'` is declared first, `%left '*' '/' '%'` second — a later declaration binds tighter, so `10+5*2` evaluates as `10+(5*2)=20`. Both levels are **left-associative**, so `20-2-3` groups as `(20-2)-3=15`.

Division and modulo by zero are rejected with `YYABORT`, which stops `yyparse()` immediately and makes it return failure.

The start rule is `S : E` with no terminator, so the whole input must be exactly one expression.

In [ ]:
%%writefile parser.y
%{
#include <stdio.h>
#include <stdlib.h>
int yylex(void);
void yyerror(const char *message);
%}

%union {
    int value;
}

%token <value> NUMBER
%type <value> E

%left '+' '-'
%left '*' '/' '%'

%%

S : E {
        printf("RESULT: %d\n", $1);
    }
  ;

E : E '+' E { $$ = $1 + $3; }
  | E '-' E { $$ = $1 - $3; }
  | E '*' E { $$ = $1 * $3; }
  | E '/' E {
        if ($3 == 0) {
            yyerror("Division by zero");
            YYABORT;
        }
        $$ = $1 / $3;
    }
  | E '%' E {
        if ($3 == 0) {
            yyerror("Modulo by zero");
            YYABORT;
        }
        $$ = $1 % $3;
    }
  | '(' E ')' { $$ = $2; }
  | NUMBER    { $$ = $1; }
  ;

%%

void yyerror(const char *message)
{
    printf("PARSER ERROR: %s\n", message);
}

int main(void)
{
    printf("Enter an arithmetic expression: ");
    if (yyparse() == 0)
        printf("RESULT: Valid expression\n");
    else
        printf("RESULT: Invalid expression\n");
    return 0;
}


## 4. Part A `lexer.l`
The scanner converts each digit string to an `int` with `atoi(yytext)` and stores it in `yylval.value` — that is how the value travels from the scanner to the parser's value stack. Any other character is returned as its character code, which the grammar rejects.

In [ ]:
%%writefile lexer.l
%option noyywrap nounput noinput
%{
#include <stdio.h>
#include <stdlib.h>
#include "parser.tab.h"
%}

%%

[0-9]+      { yylval.value = atoi(yytext); return NUMBER; }
"+"         { return '+'; }
"-"         { return '-'; }
"*"         { return '*'; }
"/"         { return '/'; }
"%"         { return '%'; }
"("         { return '('; }
")"         { return ')'; }
[ \t\r\n]+  { /* Ignore whitespace. */ }
.           { return yytext[0]; }

%%


## 5. Generate, compile and run Part A
Bison produces `parser.tab.c`/`parser.tab.h`; Flex produces `lex.yy.c`; GCC links them into one executable. The parser's `main()` and the scanner's `yylex()` end up in the same program.

In [ ]:
subprocess.run(["bison", "-d", "parser.y"], check=True)
subprocess.run(["flex", "lexer.l"], check=True)
subprocess.run(["gcc", "lex.yy.c", "parser.tab.c", "-o", "exp10a"], check=True)
print("Part A compiled.")

In [ ]:
def run_program(program, source):
    result = subprocess.run(
        ["./" + program], input=source, text=True, capture_output=True
    )
    print("INPUT:", repr(source))
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    return result.returncode

source = "10+5*2"
run_program("exp10a", source)

Expected output:
```text
INPUT: '10+5*2'
Enter an arithmetic expression: RESULT: 20
RESULT: Valid expression
```
`*` binds tighter than `+`, so this is `10+(5*2)=20`. The prompt and the first result share a line because the prompt has no newline.

In [ ]:
source = "(10+3)*2"
run_program("exp10a", source)

source = "7%3"
run_program("exp10a", source)

Expected output:
```text
INPUT: '(10+3)*2'
Enter an arithmetic expression: RESULT: 26
RESULT: Valid expression
INPUT: '7%3'
Enter an arithmetic expression: RESULT: 1
RESULT: Valid expression
```
Parentheses force `10+3` to reduce before the `*`. `%` is modulo: `7%3=1`.

In [ ]:
source = "10/0"
run_program("exp10a", source)

source = "10+"
run_program("exp10a", source)

Expected output:
```text
INPUT: '10/0'
Enter an arithmetic expression: PARSER ERROR: Division by zero
RESULT: Invalid expression
INPUT: '10+'
Enter an arithmetic expression: PARSER ERROR: syntax error
RESULT: Invalid expression
```
Division by zero is a **semantic** error — the syntax is fine, our action rejects it. A missing operand (`10+`) is a **syntax** error reported by `yyerror` as `syntax error`.

## 6. Part B — convert infix to postfix
### Grammar
```text
E -> E+T | E-T | T
T -> T*F | T/F | F
F -> (E) | -F | id
```
The layered E/T/F grammar encodes precedence by **structure**: `*` and `/` live at the tighter level T, `+` and `-` at E. No `%left` declarations are needed — the grammar is unambiguous and Bison reports no conflicts.

Each reduction **builds a postfix string**: `"%s %s +"` prints the operator *after* its operands. Unary minus becomes `NEG`.

In [ ]:
%%writefile parser.y
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
int yylex(void);
void yyerror(const char *message);
%}

%union {
    char str[256];
}

%token <str> ID
%type <str> E T F

%%

S : E {
        printf("POSTFIX: %s\n", $1);
    }
  ;

E : E '+' T {
        snprintf($$, sizeof($$), "%s %s +", $1, $3);
    }
  | E '-' T {
        snprintf($$, sizeof($$), "%s %s -", $1, $3);
    }
  | T {
        strcpy($$, $1);
    }
  ;

T : T '*' F {
        snprintf($$, sizeof($$), "%s %s *", $1, $3);
    }
  | T '/' F {
        snprintf($$, sizeof($$), "%s %s /", $1, $3);
    }
  | F {
        strcpy($$, $1);
    }
  ;

F : '(' E ')' {
        strcpy($$, $2);
    }
  | '-' F {
        snprintf($$, sizeof($$), "%s NEG", $2);
    }
  | ID {
        strcpy($$, $1);
    }
  ;

%%

void yyerror(const char *message)
{
    printf("PARSER ERROR: %s\n", message);
}

int main(void)
{
    printf("Enter an infix expression: ");
    if (yyparse() == 0)
        printf("RESULT: Valid expression\n");
    else
        printf("RESULT: Invalid expression\n");
    return 0;
}


## 7. Part B `lexer.l`
Identifiers are copied into `yylval.str` unchanged — for postfix conversion the operand itself is the semantic value; there is nothing to evaluate.

In [ ]:
%%writefile lexer.l
%option noyywrap nounput noinput
%{
#include <stdio.h>
#include <string.h>
#include "parser.tab.h"
%}

%%

[a-zA-Z_][a-zA-Z0-9_]* {
    strcpy(yylval.str, yytext);
    return ID;
}
"+"         { return '+'; }
"-"         { return '-'; }
"*"         { return '*'; }
"/"         { return '/'; }
"("         { return '('; }
")"         { return ')'; }
[ \t\r\n]+  { /* Ignore whitespace. */ }
.           { return yytext[0]; }

%%


## 8. Generate, compile and run Part B
Part B's `%%writefile` overwrites Part A's `parser.y`/`lexer.l`. To rebuild Part A afterwards, re-run its cells from Section 3.

In [ ]:
subprocess.run(["bison", "-d", "parser.y"], check=True)
subprocess.run(["flex", "lexer.l"], check=True)
subprocess.run(["gcc", "lex.yy.c", "parser.tab.c", "-o", "exp10b"], check=True)
print("Part B compiled.")

In [ ]:
source = "a+b*c"
run_program("exp10b", source)

Expected output:
```text
INPUT: 'a+b*c'
Enter an infix expression: POSTFIX: a b c * +
RESULT: Valid expression
```
`c` reduces to F then T while `a` waits, so `*` is emitted between its operands and `+` comes last.

In [ ]:
source = "(a+b)*c"
run_program("exp10b", source)

source = "-a+b"
run_program("exp10b", source)

source = "a-"
run_program("exp10b", source)

Expected output:
```text
INPUT: '(a+b)*c'
Enter an infix expression: POSTFIX: a b + c *
RESULT: Valid expression
INPUT: '-a+b'
Enter an infix expression: POSTFIX: a NEG b +
RESULT: Valid expression
INPUT: 'a-'
Enter an infix expression: PARSER ERROR: syntax error
RESULT: Invalid expression
```
The `F -> -F` rule handles unary minus; `NEG` marks it in the postfix output.

## 9. Quick viva revision

| Question | Answer |
|---|---|
| What is LEX? | A tool for generating lexical analyzers. |
| What is YACC? | A tool for generating parsers. |
| What is an infix expression? | An expression where operators appear between operands, e.g. `a+b`. |
| What is a postfix expression? | An expression where operators appear after operands, e.g. `a b +`. |
| What does `%union` declare? | The type of the parser's semantic value stack. |
| How does a token's value reach the parser? | The scanner writes it to `yylval` before returning the token. |
| What does `$$` represent? | The semantic value of the left-hand side of a grammar rule. |
| What does `$1` represent? | The semantic value of the first symbol on the right-hand side. |
| Why use `%left`? | To specify left associativity and operator precedence (Part A). |
| Why no `%left` in Part B? | The layered E/T/F grammar already encodes precedence; there are no conflicts. |
| What does `YYABORT` do? | Stops `yyparse()` immediately and returns failure (used for division by zero). |
| Which precedence does `a+b*c` use? | `*` before `+`, so the postfix is `a b c * +`. |

## 10. Key points
- **10A** evaluates integer constants only; identifiers are not part of its grammar.
- **10B** treats identifiers such as `a` and `b` as operands and rearranges them — it never evaluates.
- Both parts share the same source file names: **Part B overwrites Part A's** `parser.y`/`lexer.l`; re-run Section 3 to rebuild Part A.